In [40]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

IMAGES_DIR = Path.cwd().parent / ("docs/images")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [10]:
DATA_DIR = Path.cwd().parent / "data"

files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
}

dfs = {name: pd.read_csv(DATA_DIR / fname) for name, fname in files.items()}

for name, df in dfs.items():
    print(f"\n=== {name} ({df.shape}) ===")
    print(df.dtypes)


=== orders ((99441, 8)) ===
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

=== order_items ((112650, 7)) ===
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

=== order_payments ((103886, 5)) ===
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

=== order_reviews ((99224, 7)) ===
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_

In [11]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

dfs["orders"][date_cols] = dfs["orders"][date_cols].apply(pd.to_datetime)

In [18]:
print("=== NA ===")
print(dfs["orders"].isna().sum())
print("=== Order Status ===")
print(dfs["orders"]["order_status"].value_counts())

=== NA ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
=== Order Status ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [19]:
delivered = dfs["orders"][dfs["orders"]["order_delivered_customer_date"].notna()].copy()
delivered["delay_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.days
delivered["is_late"] = delivered["delay_days"] > 0

In [20]:
merged = delivered.merge(dfs["order_reviews"][["order_id", "review_score"]], on="order_id", how="left")
merged.groupby("is_late")["review_score"].mean()

is_late
False    4.289842
True     2.271139
Name: review_score, dtype: float64

In [26]:
late_rate = merged["is_late"].mean()
print(late_rate)

0.06765630637596


In [27]:
by_state = merged.merge(dfs["customers"][["customer_id", "customer_state"]], on="customer_id") \
    .groupby("customer_state")["is_late"].mean().sort_values(ascending=False)
print(by_state)

customer_state
AL    0.214464
MA    0.173370
SE    0.152239
PI    0.138365
CE    0.138066
BA    0.122212
RR    0.121951
RJ    0.120985
PA    0.111345
ES    0.106786
PB    0.104247
TO    0.098540
MS    0.095506
PE    0.095387
RN    0.092437
SC    0.082280
GO    0.065856
RS    0.061141
MT    0.059618
DF    0.056217
MG    0.045693
SP    0.044785
PR    0.040469
AC    0.037500
AP    0.029851
RO    0.028807
AM    0.027397
Name: is_late, dtype: float64


In [28]:
by_category = merged.merge(dfs["order_items"][["order_id", "product_id"]], on="order_id") \
.merge(dfs["products"][["product_id", "product_category_name"]], on="product_id").groupby("product_category_name")["is_late"].mean().sort_values(ascending=False)
print(by_category)

product_category_name
moveis_colchao_e_estofado         0.135135
casa_conforto_2                   0.133333
audio                             0.115702
artigos_de_natal                  0.100000
fashion_underwear_e_moda_praia    0.094488
                                    ...   
artigos_de_festas                 0.000000
seguros_e_servicos                0.000000
fashion_roupa_infanto_juvenil     0.000000
fraldas_higiene                   0.000000
pc_gamer                          0.000000
Name: is_late, Length: 73, dtype: float64


In [ ]:
merged.groupby("is_late")["review_score"].agg(["mean", "count"])
merged["delay_days"].corr(merged["review_score"])

np.float64(-0.26676366163799653)

In [42]:
fig, ax = plt.subplots(figsize=(8, 6))
by_state.head(10).plot(kind="barh", ax=ax)
ax.set_xlabel("Late Delivery Rate")
ax.set_ylabel("State")
ax.set_title("Top 10 States by Late Delivery Rate")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "late_rate_by_state.png", dpi=150)
plt.close()

In [43]:
fig, ax = plt.subplots(figsize=(8, 6))
by_category.head(10).plot(kind="barh", ax=ax, color="orange")
ax.set_xlabel("Late Delivery Rate")
ax.set_ylabel("Product Category")
ax.set_title("Top 10 Categories by Late Delivery Rate")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "late_rate_by_category.png", dpi=150)
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=merged, x="review_score", y="delay_days", ax=ax)
ax.axhline(0, color="red", linestyle="--", linewidth=1)  # 0일 = 정시 기준선
ax.set_title("Delivery Delay Distribution by Review Score")
ax.set_xlabel("Review Score")
ax.set_ylabel("Delay (days, negative = early)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "delay_vs_review_score.png", dpi=150)
plt.close()

![Late delivery rate by state](../docs/images/late_rate_by_state.png)
![Late delivery rate by category](../docs/images/late_rate_by_category.png)
![Delay vs review score](../docs/images/delay_vs_review_score.png)